## Modelo educacional pra Series Temporais 

- 5 famílias de modelo

cnn, gru, lstm, cnn_gru, cnn_lstm

- 3 tamanhos

small, medium, large

- estratégia de:

    - config como dicionário

    - pasta por experimento

    - salvar config.json, metrics.json, modelo

    - logar tudo em um results.csv

In [18]:
import os
import tensorflow as tf

# Configuração para GPU (NVIDIA ou Metal)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPUs detectadas: {len(gpus)}")
        print(f"GPU: {gpus}")
    except RuntimeError as e:
        print(e)
else:
    print("Nenhuma GPU detectada. Usando CPU.")

RuntimeError: Visible devices cannot be modified after being initialized

In [ ]:
# APENAS PARA COLAB: montar Google Drive
import os
if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive montado em /content/drive")
else:
    print("Rodando localmente (não é Colab)")

1. Imports e setup

In [1]:
import os
import json
from datetime import datetime

import numpy as np
import pandas as pd
import h5py
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import models, layers
from tensorflow.keras.layers import (
    Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout,
    GRU, LSTM, Bidirectional
)

BASE_EXPERIMENTS_DIR = "experiments_ts_models"
os.makedirs(BASE_EXPERIMENTS_DIR, exist_ok=True)

print("TensorFlow:", tf.__version__)


TensorFlow: 2.10.0


2. Funções de utilidade (pasta do experimento + CSV)

In [2]:
def create_run_dir(config, base_dir=BASE_EXPERIMENTS_DIR):
    """
    Cria uma pasta única para cada experimento, baseada em timestamp + família + tamanho.
    """
    os.makedirs(base_dir, exist_ok=True)
    ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    
    family = config.get("family", "na")
    size = config.get("size", "na")
    lr = config.get("lr", "na")
    
    run_name = f"{ts}_{family}_{size}_lr={lr}"
    run_dir = os.path.join(base_dir, run_name)
    os.makedirs(run_dir, exist_ok=True)
    return run_dir


def log_run_summary(config, metrics, run_dir,
                    results_path=os.path.join(BASE_EXPERIMENTS_DIR, "results.csv")):
    """
    Adiciona linha no CSV de resultados: config + métricas + run_dir.
    """
    row = {**config, **metrics, "run_dir": run_dir}
    
    if os.path.exists(results_path):
        df = pd.read_csv(results_path)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    else:
        df = pd.DataFrame([row])
    
    df.to_csv(results_path, index=False)
    print(f"[LOG] Run registrado em {results_path}")


3. Dataset loading/spliting

In [3]:
#load X from H5 file
input_path = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_pre files\X_normalized.h5"
with h5py.File(input_path, "r") as f:
    X = f["data"][:]  # Load the dataset into a NumPy array

X = X.astype("float32")

# print shape of X
print("X shape:", X.shape)

X shape: (4400, 9000, 8)


In [4]:
# Load subject metadata
meta = pd.read_csv(r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_pre files\X_train_h7ipJUo.csv")    # contains subject IDs
print(meta.head())

subject_ids = meta["Subject_ID"].to_numpy()

# Load y CSV file
y_path = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_pre files\y_train_tX9Br0C.csv"


y_df = pd.read_csv(y_path)

# Save ID separately if needed for alignment or merging
y_id = y_df["ID"].to_numpy()

# Keep only mask columns
mask_cols = [c for c in y_df.columns if c.startswith("y_")]

y = y_df[mask_cols].to_numpy()
# Convert y to float32
y = y.astype("float32")




print("y shape:", y.shape)
print("Subjects shape:", subject_ids.shape)


    ID  Subject_ID
0  0.0         0.0
1  1.0         0.0
2  2.0         0.0
3  3.0         0.0
4  4.0         0.0
y shape: (4400, 90)
Subjects shape: (4400,)


Subject-wise Train/Validation Split

In [5]:
def train_val_split_by_subject(X, y, subject_ids, train_ratio=0.7, seed=42):
    rng = np.random.default_rng(seed)

    unique_subj = np.unique(subject_ids)
    rng.shuffle(unique_subj)

    n_train = int(len(unique_subj) * train_ratio)
    train_subj = unique_subj[:n_train]

    train_mask = np.isin(subject_ids, train_subj)
    val_mask   = ~train_mask

    return X[train_mask], X[val_mask], y[train_mask], y[val_mask], train_subj


X_train, X_val, y_train, y_val, train_subjects = train_val_split_by_subject(
    X, y, subject_ids, train_ratio=0.7, seed=42
)

In [7]:
def make_datasets(batch_size):
    # Dataset de treino
    train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    train_ds = train_ds.shuffle(buffer_size=len(X_train)).batch(batch_size)
    
    # Dataset de validação
    val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    val_ds = val_ds.batch(batch_size)
    
    return train_ds, val_ds

4. As 5 famílias × 3 tamanhos (builder de modelos)

Aqui entra o “cardápio” organizado por family + size.

4.1. CNN 1D

In [8]:
def build_cnn_small(input_shape):
    model = models.Sequential([
        Conv1D(32, 5, activation="relu", input_shape=input_shape),
        GlobalAveragePooling1D(),
        Dense(1)  # regressão (MSE); se fosse classificação -> activation="sigmoid"
    ])
    return model


def build_cnn_medium(input_shape):
    model = models.Sequential([
        Conv1D(32, 5, activation="relu", input_shape=input_shape),
        Conv1D(64, 5, activation="relu"),
        MaxPooling1D(2),
        Conv1D(64, 5, activation="relu"),
        GlobalAveragePooling1D(),
        Dense(64, activation="relu"),
        Dense(1)
    ])
    return model


def build_cnn_large(input_shape):
    model = models.Sequential([
        Conv1D(64, 7, activation="relu", input_shape=input_shape),
        Conv1D(64, 5, activation="relu"),
        MaxPooling1D(2),
        Dropout(0.3),
        Conv1D(128, 5, activation="relu"),
        Conv1D(128, 3, activation="relu"),
        MaxPooling1D(2),
        Dropout(0.3),
        GlobalAveragePooling1D(),
        Dense(128, activation="relu"),
        Dropout(0.3),
        Dense(1)
    ])
    return model


4.2. GRU

In [ ]:
def build_gru_small(input_shape):
    model = models.Sequential([
        GRU(32, input_shape=input_shape),
        Dense(1)
    ])
    return model


def build_gru_medium(input_shape):
    model = models.Sequential([
        GRU(64, return_sequences=True, input_shape=input_shape),
        GRU(32),
        Dense(32, activation="relu"),
        Dense(1)
    ])
    return model


def build_gru_large(input_shape):
    model = models.Sequential([
        Bidirectional(GRU(128, return_sequences=True), input_shape=input_shape),
        GRU(64),
        Dense(128, activation="relu"),
        Dropout(0.3),
        Dense(64, activation="relu"),
        Dense(1)
    ])
    return model

4.3. LSTM

In [ ]:
def build_lstm_small(input_shape):
    model = models.Sequential([
        LSTM(32, input_shape=input_shape),
        Dense(1)
    ])
    return model


def build_lstm_medium(input_shape):
    model = models.Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        LSTM(32),
        Dense(32, activation="relu"),
        Dense(1)
    ])
    return model


def build_lstm_large(input_shape):
    model = models.Sequential([
        Bidirectional(LSTM(128, return_sequences=True), input_shape=input_shape),
        LSTM(64),
        Dense(128, activation="relu"),
        Dropout(0.3),
        Dense(64, activation="relu"),
        Dense(1)
    ])
    return model

4.4. CNN + GRU

In [ ]:
def build_cnn_gru_small(input_shape):
    model = models.Sequential([
        Conv1D(32, 5, activation="relu", input_shape=input_shape),
        MaxPooling1D(2),
        GRU(32),
        Dense(1)
    ])
    return model


def build_cnn_gru_medium(input_shape):
    model = models.Sequential([
        Conv1D(32, 5, activation="relu", input_shape=input_shape),
        Conv1D(64, 5, activation="relu"),
        MaxPooling1D(2),
        GRU(64),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(1)
    ])
    return model


def build_cnn_gru_large(input_shape):
    model = models.Sequential([
        Conv1D(64, 7, activation="relu", input_shape=input_shape),
        Conv1D(64, 5, activation="relu"),
        MaxPooling1D(2),
        Dropout(0.3),
        Conv1D(128, 5, activation="relu"),
        MaxPooling1D(2),
        Bidirectional(GRU(128)),
        Dense(128, activation="relu"),
        Dropout(0.3),
        Dense(64, activation="relu"),
        Dense(1)
    ])
    return model

4.5. CNN + LSTM

In [ ]:
def build_cnn_lstm_small(input_shape):
    model = models.Sequential([
        Conv1D(32, 5, activation="relu", input_shape=input_shape),
        MaxPooling1D(2),
        LSTM(32),
        Dense(1)
    ])
    return model


def build_cnn_lstm_medium(input_shape):
    model = models.Sequential([
        Conv1D(32, 5, activation="relu", input_shape=input_shape),
        Conv1D(64, 5, activation="relu"),
        MaxPooling1D(2),
        LSTM(64),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(1)
    ])
    return model


def build_cnn_lstm_large(input_shape):
    model = models.Sequential([
        Conv1D(64, 7, activation="relu", input_shape=input_shape),
        Conv1D(64, 5, activation="relu"),
        MaxPooling1D(2),
        Dropout(0.3),
        Conv1D(128, 5, activation="relu"),
        MaxPooling1D(2),
        Bidirectional(LSTM(128)),
        Dense(128, activation="relu"),
        Dropout(0.3),
        Dense(64, activation="relu"),
        Dense(1)
    ])
    return model

5. Router geral: build_model_from_config

Em vez de você escolher o modelo “na mão”, usa family + size no config e essa função decide.

In [13]:
def build_model_from_config(config, input_shape):
    family = config["family"]      # "cnn", "gru", "lstm", "cnn_gru", "cnn_lstm"
    size = config["size"]          # "small", "medium", "large"
    
    if family == "cnn":
        if size == "small":
            return build_cnn_small(input_shape)
        elif size == "medium":
            return build_cnn_medium(input_shape)
        else:
            return build_cnn_large(input_shape)
    
    elif family == "gru":
        if size == "small":
            return build_gru_small(input_shape)
        elif size == "medium":
            return build_gru_medium(input_shape)
        else:
            return build_gru_large(input_shape)
    
    elif family == "lstm":
        if size == "small":
            return build_lstm_small(input_shape)
        elif size == "medium":
            return build_lstm_medium(input_shape)
        else:
            return build_lstm_large(input_shape)
    
    elif family == "cnn_gru":
        if size == "small":
            return build_cnn_gru_small(input_shape)
        elif size == "medium":
            return build_cnn_gru_medium(input_shape)
        else:
            return build_cnn_gru_large(input_shape)
    
    elif family == "cnn_lstm":
        if size == "small":
            return build_cnn_lstm_small(input_shape)
        elif size == "medium":
            return build_cnn_lstm_medium(input_shape)
        else:
            return build_cnn_lstm_large(input_shape)
    
    else:
        raise ValueError(f"Família desconhecida: {family}")


6. Função única de treino + salvamento (core educacional)

Essa aqui junta tudo:

usa config como fonte da verdade

cria pasta do experimento

salva config.json

treina modelo

calcula métricas finais

salva metrics.json

salva modelo

loga no results.csv

In [14]:
def train_and_evaluate(config):
    # 1. Seeds
    seed = config.get("random_seed", 42)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    
    lookback = config["lookback"]
    batch_size = config["batch_size"]
    epochs = config["epochs"]
    lr = config["lr"]
    
    # 2. Datasets a partir de X_train / y_train reais
    train_ds, val_ds = make_datasets(batch_size)

    # 3. Descobrir input_shape a partir de um batch
    for xb, yb in train_ds.take(1):
        input_shape = xb.shape[1:]   # (9000, 8) no seu caso
    print("Input shape:", input_shape)
    
    # 4. Pasta do experimento
    run_dir = create_run_dir(config)
    print(f"\n[RUN] {run_dir}")
    
    # 5. Salvar config.json
    config_path = os.path.join(run_dir, "config.json")
    with open(config_path, "w") as f:
        json.dump(config, f, indent=4)
    print(f"[SAVE] config.json")
    
    # 6. Modelo
    model = build_model_from_config(config, input_shape=input_shape)
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss="mse", metrics=["mae"])
    
    model.summary()
    
    # 7. Treino
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        verbose=1
    )
    
    # 8. Métricas finais
    train_loss = history.history["loss"][-1]
    val_loss = history.history["val_loss"][-1]
    train_mae = history.history["mae"][-1]
    val_mae = history.history["val_mae"][-1]
    
    metrics = {
        "final_train_loss": float(train_loss),
        "final_val_loss": float(val_loss),
        "final_train_mae": float(train_mae),
        "final_val_mae": float(val_mae),
    }
    
    # 9. Salvar metrics.json
    metrics_path = os.path.join(run_dir, "metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=4)
    print(f"[SAVE] metrics.json")
    
    # 10. Salvar modelo
    model_path = os.path.join(run_dir, "model.keras")
    model.save(model_path)
    print(f"[SAVE] model.keras")
    
    # 11. Log geral
    log_run_summary(config, metrics, run_dir)
    
    print(f"[METRICS] val_loss={val_loss:.4f} | val_mae={val_mae:.4f}")
    
    return model, metrics, run_dir


7. Montando a grade de experimentos (small/medium/large × famílias)

Agora vem a parte legal: você só define dicionários de config, e o resto é automático.

Exemplo com alguns modelos (você pode completar depois):

In [15]:
experiments_CNN = [
    {
         "experiment_name": "cnn_small",
         "family": "cnn",
         "size": "small",
         "random_seed": 42,
         "lookback": 30,
         "batch_size": 32,
         "lr": 0.001,
         "epochs": 3,
    },
    {
        "experiment_name": "cnn_medium",
        "family": "cnn",
        "size": "medium",
        "random_seed": 42,
        "lookback": 30,
        "batch_size": 32,
        "lr": 0.001,
        "epochs": 3,
    },     
    {
        "experiment_name": "cnn_large",
        "family": "cnn",
        "size": "large",
        "random_seed": 42,
        "lookback": 30,
        "batch_size": 32,
        "lr": 0.001,
        "epochs": 3,
    },
]

In [ ]:
for cfg in experiments_CNN:
    model, metrics, run_dir = train_and_evaluate(cfg)


Input shape: (9000, 8)

[RUN] experiments_ts_models\2025-12-11_21-46-43_cnn_small_lr=0.001
[SAVE] config.json
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d (Conv1D)             (None, 8996, 32)          1312      
                                                                 
 global_average_pooling1d (G  (None, 32)               0         
 lobalAveragePooling1D)                                          
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 1,345
Trainable params: 1,345
Non-trainable params: 0
_________________________________________________________________
Epoch 1/3
94/94 [==============================] - 8s 78ms/step - loss: 0.0792 - mae: 0.1524 - val_loss: 0.0771 - val_mae: 0.1327
Epoch 2/3


In [16]:
experiments_GRU = [
        {
        "experiment_name": "gru_small",
        "family": "gru",
        "size": "small",
        "random_seed": 42,
        "lookback": 30,
        "batch_size": 32,
        "lr": 0.001,
        "epochs": 3,
        },
        {
        "experiment_name": "gru_medium",
        "family": "gru",
        "size": "medium",
        "random_seed": 42,
        "lookback": 30,
        "batch_size": 32,
        "lr": 0.001,
        "epochs": 3,
        },
        {
        "experiment_name": "gru_large",
        "family": "gru",
        "size": "large",
        "random_seed": 42,
        "lookback": 30,
        "batch_size": 32,
        "lr": 0.001,
        "epochs": 3,
        },
]    


In [17]:
for cfg in experiments_GRU:
    model, metrics, run_dir = train_and_evaluate(cfg)

Input shape: (9000, 8)

[RUN] experiments_ts_models\2025-12-11_21-50-13_gru_small_lr=0.001
[SAVE] config.json
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 gru (GRU)                   (None, 32)                4032      
                                                                 
 dense (Dense)               (None, 1)                 33        
                                                                 
Total params: 4,065
Trainable params: 4,065
Non-trainable params: 0
_________________________________________________________________
Epoch 1/3


InvalidArgumentError: Graph execution error:

No OpKernel was registered to support Op 'CudnnRNN' used by {{node CudnnRNN}} with these attrs: [seed=0, dropout=0, T=DT_FLOAT, input_mode="linear_input", direction="unidirectional", rnn_mode="gru", seed2=0, is_training=true]
Registered devices: [CPU, GPU]
Registered kernels:
  <no registered kernels>

	 [[CudnnRNN]]
	 [[sequential/gru/PartitionedCall]] [Op:__inference_train_function_2993]

In [ ]:
experiments_LSTM = [
        {
        "experiment_name": "lstm_small",
        "family": "lstm",
        "size": "small",
        "random_seed": 42,
        "lookback": 30,
        "batch_size": 32,
        "lr": 0.001,
        "epochs": 3,
        },
        {
        "experiment_name": "lstm_medium",
        "family": "lstm",
        "size": "medium",
        "random_seed": 42,
        "lookback": 30,
        "batch_size": 32,
        "lr": 0.001,
        "epochs": 3,
        },
        {
        "experiment_name": "lstm_large",
        "family": "lstm",
        "size": "large",
        "random_seed": 42,
        "lookback": 30,
        "batch_size": 32,
        "lr": 0.001,
        "epochs": 3,
        },  

]




In [ ]:
for cfg in experiments_LSTM:
    model, metrics, run_dir = train_and_evaluate(cfg)

In [ ]:
experiments_CNN_GRU = [
        {
        "experiment_name": "cnn_gru_small",
        "family": "cnn_gru",
        "size": "small",
        "random_seed": 42,
        "lookback": 30,
        "batch_size": 32,
        "lr": 0.001,
        "epochs": 3,  
        },
        {
        "experiment_name": "cnn_gru_medium",
        "family": "cnn_gru",
        "size": "medium",
        "random_seed": 42,
        "lookback": 30,
        "batch_size": 32,
        "lr": 0.001,
        "epochs": 3,  
        },
        {
        "experiment_name": "cnn_gru_large",
        "family": "cnn_gru",
        "size": "large",
        "random_seed": 42,
        "lookback": 30,
        "batch_size": 32,
        "lr": 0.001,
        "epochs": 3,  
        },  
]


In [ ]:
for cfg in experiments_CNN_GRU:
    model, metrics, run_dir = train_and_evaluate(cfg)

In [ ]:
experiments_CNN_LSTM = [
        {
        "experiment_name": "cnn_lstm_small",
        "family": "cnn_lstm",
        "size": "small",
        "random_seed": 42,
        "lookback": 30,
        "batch_size": 32,
        "lr": 0.001,
        "epochs": 3,  
        },
        {
        "experiment_name": "cnn_lstm_medium",
        "family": "cnn_lstm",
        "size": "medium",
        "random_seed": 42,
        "lookback": 30,
        "batch_size": 32,
        "lr": 0.001,
        "epochs": 3,  
        },
        {
        "experiment_name": "cnn_lstm_large",
        "family": "cnn_lstm",
        "size": "large",
        "random_seed": 42,
        "lookback": 30,
        "batch_size": 32,
        "lr": 0.001,
        "epochs": 3,
        },  
]

In [ ]:
for cfg in experiments_CNN_LSTM:
    model, metrics, run_dir = train_and_evaluate(cfg)


KeyboardInterrupt: 

In [ ]:
8. Olhando o results.csv depois

In [ ]:
results_path = os.path.join(BASE_EXPERIMENTS_DIR, "results.csv")

df = pd.read_csv(results_path)
df_sorted = df.sort_values(by="final_val_loss", ascending=True)
df_sorted
